# 3.4_mini_agent_loop

In [1]:
# User input
#    ↓
# LLM 判断是否需要调用 tool
#    ↓
# 如果需要：解析 tool_call
#    ↓
# Python 执行对应工具函数
#    ↓
# 把 tool result 回传给模型
#    ↓
# 模型生成最终回答

## 实现一个可以管理学习任务的 Mini Agent Loop
最小功能：
1. 用户输入自然语言
2. 模型判断是否需要调用 task tools
3. 如果需要，解析 tool_call
4. Python 执行对应函数
5. 把 tool result 作为 tool message 回传给模型
6. 模型生成最终回答

## 导入依赖

In [2]:
from __future__ import annotations

import json
import os
from dataclasses import asdict, dataclass
from typing import Any, Callable, Literal, TypedDict, cast

from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import ChatCompletionMessageParam, ChatCompletionToolParam

## 配置 DeepSeek Client

In [3]:
load_dotenv()

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

if not DEEPSEEK_API_KEY:
    raise RuntimeError(
        "Missing DEEPSEEK_API_KEY. Please add it to your .env file."
    )

client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com",
)

# 你之前使用的是 deepseek-v4-flash，所以这里默认保留。
# 如果你的账号实际模型名不同，可以在 .env 里配置：
# DEEPSEEK_MODEL=你的模型名
MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash")

print("MODEL:", MODEL)

MODEL: deepseek-v4-flash


## 定义任务类型

In [4]:
TaskStatus = Literal["todo", "doing", "done"]
TaskPriority = Literal["low", "medium", "high"]


VALID_STATUSES: set[str] = {"todo", "doing", "done"}
VALID_PRIORITIES: set[str] = {"low", "medium", "high"}


@dataclass
class Task:
    id: int
    title: str
    status: TaskStatus
    priority: TaskPriority


TaskDict = dict[str, Any]
ToolResult = dict[str, Any]

## 定义本地任务数据库

In [5]:
@dataclass
class TaskStore:
    tasks: dict[int, Task]
    next_task_id: int


store = TaskStore(
    tasks={},
    next_task_id=1,
)


def reset_task_store() -> None:
    """
    Reset local task storage.

    这个函数只是为了 notebook 反复测试方便。
    正式项目里面可以换成 SQLite / JSON 文件 / PostgreSQL。
    """
    store.tasks.clear()
    store.next_task_id = 1


def task_to_dict(task: Task) -> TaskDict:
    """
    Convert Task dataclass to plain dict.

    LLM tool result 最好返回 JSON-serializable 的 dict，
    不要直接返回 dataclass object。
    """
    return asdict(task)


def get_all_tasks_as_dicts() -> list[TaskDict]:
    """
    Debug helper.

    用来在 notebook 里面直接查看当前所有任务。
    """
    return [task_to_dict(task) for task in store.tasks.values()]

## 实现工具函数

In [6]:
def normalize_priority(priority: str | None) -> TaskPriority:
    """
    Normalize priority from model arguments.

    模型有时候可能不给 priority，或者给出非法值。
    所以工具函数自己要做 runtime validation。
    """
    if priority is None:
        return "medium"

    priority = priority.lower().strip()

    if priority not in VALID_PRIORITIES:
        raise ValueError(
            f"Invalid priority: {priority}. "
            f"Expected one of: {sorted(VALID_PRIORITIES)}"
        )

    return cast(TaskPriority, priority)


def normalize_status(status: str | None) -> TaskStatus | None:
    """
    Normalize status from model arguments.

    list_tasks 允许 status=None，表示查询所有任务。
    """
    if status is None:
        return None

    status = status.lower().strip()

    if status == "all":
        return None

    if status not in VALID_STATUSES:
        raise ValueError(
            f"Invalid status: {status}. "
            f"Expected one of: {sorted(VALID_STATUSES)}"
        )

    return cast(TaskStatus, status)


def add_task(title: str, priority: str = "medium") -> ToolResult:
    """
    Add a new study task.

    这是一个会修改外部状态的工具。
    """
    title = title.strip()

    if not title:
        return {
            "ok": False,
            "error": "Task title cannot be empty.",
        }

    try:
        normalized_priority = normalize_priority(priority)
    except ValueError as error:
        return {
            "ok": False,
            "error": str(error),
        }

    task = Task(
        id=store.next_task_id,
        title=title,
        status="todo",
        priority=normalized_priority,
    )

    store.tasks[task.id] = task
    store.next_task_id += 1

    return {
        "ok": True,
        "message": "Task added successfully.",
        "task": task_to_dict(task),
    }


def list_tasks(status: str | None = None) -> ToolResult:
    """
    List tasks.

    status:
    - None / all: return all tasks
    - todo: return todo tasks
    - doing: return doing tasks
    - done: return done tasks
    """
    try:
        normalized_status = normalize_status(status)
    except ValueError as error:
        return {
            "ok": False,
            "error": str(error),
        }

    tasks = list(store.tasks.values())

    if normalized_status is not None:
        tasks = [task for task in tasks if task.status == normalized_status]

    return {
        "ok": True,
        "count": len(tasks),
        "tasks": [task_to_dict(task) for task in tasks],
    }


def update_task_status(task_id: int, status: str) -> ToolResult:
    """
    Update task status by task id.

    这是另一个会修改外部状态的工具。
    """
    try:
        task_id = int(task_id)
    except ValueError:
        return {
            "ok": False,
            "error": f"Invalid task_id: {task_id}. Expected an integer.",
        }

    if task_id not in store.tasks:
        return {
            "ok": False,
            "error": f"Task {task_id} does not exist.",
        }

    try:
        normalized_status = normalize_status(status)
    except ValueError as error:
        return {
            "ok": False,
            "error": str(error),
        }

    if normalized_status is None:
        return {
            "ok": False,
            "error": "Status cannot be all or None when updating a task.",
        }

    task = store.tasks[task_id]
    task.status = normalized_status

    return {
        "ok": True,
        "message": "Task status updated successfully.",
        "task": task_to_dict(task),
    }


def search_tasks(keyword: str) -> ToolResult:
    """
    Search tasks by keyword.

    这个工具不修改状态，只负责查询。
    """
    keyword = keyword.strip().lower()

    if not keyword:
        return {
            "ok": False,
            "error": "Keyword cannot be empty.",
        }

    matched_tasks = [
        task
        for task in store.tasks.values()
        if keyword in task.title.lower()
    ]

    return {
        "ok": True,
        "keyword": keyword,
        "count": len(matched_tasks),
        "tasks": [task_to_dict(task) for task in matched_tasks],
    }

## 定义 Tool Schema

In [7]:
TOOLS: list[ChatCompletionToolParam] = [
    cast(
        ChatCompletionToolParam,
        {
            "type": "function",
            "function": {
                "name": "add_task",
                "description": "Add a new study task to the task list.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "title": {
                            "type": "string",
                            "description": "The title of the task.",
                        },
                        "priority": {
                            "type": "string",
                            "enum": ["low", "medium", "high"],
                            "description": "The priority of the task.",
                        },
                    },
                    "required": ["title"],
                },
            },
        },
    ),
    cast(
        ChatCompletionToolParam,
        {
            "type": "function",
            "function": {
                "name": "list_tasks",
                "description": "List study tasks. Can optionally filter by status.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "status": {
                            "type": "string",
                            "enum": ["todo", "doing", "done", "all"],
                            "description": (
                                "Task status filter. Use all if the user wants "
                                "to see every task."
                            ),
                        },
                    },
                    "required": [],
                },
            },
        },
    ),
    cast(
        ChatCompletionToolParam,
        {
            "type": "function",
            "function": {
                "name": "update_task_status",
                "description": "Update the status of a task by task id.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "task_id": {
                            "type": "integer",
                            "description": "The id of the task to update.",
                        },
                        "status": {
                            "type": "string",
                            "enum": ["todo", "doing", "done"],
                            "description": "The new status of the task.",
                        },
                    },
                    "required": ["task_id", "status"],
                },
            },
        },
    ),
    cast(
        ChatCompletionToolParam,
        {
            "type": "function",
            "function": {
                "name": "search_tasks",
                "description": "Search study tasks by keyword.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "keyword": {
                            "type": "string",
                            "description": "Keyword to search in task titles.",
                        },
                    },
                    "required": ["keyword"],
                },
            },
        },
    ),
]

## 建立 Function Dispatch

In [8]:
ToolFunction = Callable[..., ToolResult]


TOOL_MAP: dict[str, ToolFunction] = {
    "add_task": add_task,
    "list_tasks": list_tasks,
    "update_task_status": update_task_status,
    "search_tasks": search_tasks,
}

## 定义 Agent 的 System Prompt 和 Messages

In [9]:
SYSTEM_PROMPT = """
You are a Mini Study Task Agent.

Your job is to help the user manage study tasks related to SWE Agent learning.

You have access to task management tools:
- add_task
- list_tasks
- update_task_status
- search_tasks

Rules:
1. If the user asks conceptual questions, answer directly without using tools.
2. If the user wants to add, list, update, or search tasks, use the appropriate tool.
3. Do not pretend that a task was added, updated, or found unless the tool result confirms it.
4. After receiving tool results, explain the result clearly in Chinese.
5. Keep the final answer concise and practical.
""".strip()


messages: list[ChatCompletionMessageParam] = [
    cast(
        ChatCompletionMessageParam,
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
    )
]


def reset_agent_messages() -> None:
    """
    Reset conversation messages.

    这个函数不会清空 task_store。
    如果你要同时清空任务，需要另外调用 reset_task_store()。
    """
    global messages

    messages = [
        cast(
            ChatCompletionMessageParam,
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
        )
    ]

## Tool Call 解析与执行

In [10]:
class ToolExecution(TypedDict):
    tool_call_id: str
    tool_name: str
    arguments: dict[str, Any]
    result: ToolResult


def parse_tool_arguments(raw_arguments: str | None) -> dict[str, Any]:
    """
    Parse tool call arguments from JSON string.

    模型返回的 function.arguments 通常是 JSON string。
    所以这里要先 json.loads。
    """
    if raw_arguments is None or raw_arguments.strip() == "":
        return {}

    try:
        parsed = json.loads(raw_arguments)
    except json.JSONDecodeError as error:
        raise ValueError(f"Invalid JSON arguments: {raw_arguments}") from error

    if not isinstance(parsed, dict):
        raise ValueError(
            f"Tool arguments must be a JSON object, got: {type(parsed)}"
        )

    return cast(dict[str, Any], parsed)


def execute_tool_call(tool_call: Any) -> ToolExecution:
    """
    Execute one tool call returned by the model.

    注意：
    这里 deliberately 使用 Any + getattr，
    是为了避免 Pylance 对 OpenAI SDK tool_call union type 的误报。
    之前你遇到过：
    Cannot access attribute "function" for class "ChatCompletionMessageCustomToolCall"

    用 getattr 可以绕开这个类型问题，同时保留 runtime 安全检查。
    """
    tool_call_id = str(getattr(tool_call, "id", ""))

    function_obj = getattr(tool_call, "function", None)

    if function_obj is None:
        return {
            "tool_call_id": tool_call_id,
            "tool_name": "",
            "arguments": {},
            "result": {
                "ok": False,
                "error": "This tool call does not contain a function object.",
            },
        }

    tool_name = str(getattr(function_obj, "name", ""))
    raw_arguments = getattr(function_obj, "arguments", "{}")

    try:
        arguments = parse_tool_arguments(raw_arguments)
    except ValueError as error:
        return {
            "tool_call_id": tool_call_id,
            "tool_name": tool_name,
            "arguments": {},
            "result": {
                "ok": False,
                "error": str(error),
            },
        }

    tool_function = TOOL_MAP.get(tool_name)

    if tool_function is None:
        return {
            "tool_call_id": tool_call_id,
            "tool_name": tool_name,
            "arguments": arguments,
            "result": {
                "ok": False,
                "error": f"Unknown tool: {tool_name}",
            },
        }

    try:
        result = tool_function(**arguments)
    except TypeError as error:
        result = {
            "ok": False,
            "error": (
                f"Tool argument mismatch for {tool_name}: {str(error)}"
            ),
        }
    except Exception as error:
        result = {
            "ok": False,
            "error": (
                f"Unexpected error while running {tool_name}: {str(error)}"
            ),
        }

    return {
        "tool_call_id": tool_call_id,
        "tool_name": tool_name,
        "arguments": arguments,
        "result": result,
    }

## 把 Assistant Message 转成可再次传入 API 的格式

In [11]:
def assistant_message_to_param(message: Any) -> ChatCompletionMessageParam:
    """
    Convert SDK response message object to ChatCompletionMessageParam.

    为什么要自己转换？

    因为 API 返回的是 SDK object，
    但下一轮 messages 需要的是 dict-like message param。

    直接 messages.append(response.choices[0].message)
    可能能运行，但 Pylance 经常会报类型错误。
    所以这里手动构造一个干净的 dict。
    """
    message_param: dict[str, Any] = {
        "role": "assistant",
    }

    content = getattr(message, "content", None)
    if content is not None:
        message_param["content"] = content

    tool_calls = getattr(message, "tool_calls", None)

    if tool_calls:
        converted_tool_calls: list[dict[str, Any]] = []

        for tool_call in tool_calls:
            function_obj = getattr(tool_call, "function", None)

            if function_obj is None:
                continue

            converted_tool_calls.append(
                {
                    "id": getattr(tool_call, "id", ""),
                    "type": getattr(tool_call, "type", "function"),
                    "function": {
                        "name": getattr(function_obj, "name", ""),
                        "arguments": getattr(function_obj, "arguments", "{}"),
                    },
                }
            )

        message_param["tool_calls"] = converted_tool_calls

    return cast(ChatCompletionMessageParam, message_param)


def tool_result_to_message(
    tool_call_id: str,
    result: ToolResult,
) -> ChatCompletionMessageParam:
    """
    Convert local Python tool result to tool message.

    这是 Agent Loop 里面非常关键的一步：

    Python 执行工具
        ↓
    result 转成 JSON string
        ↓
    作为 role='tool' 的 message 回传给模型
    """
    return cast(
        ChatCompletionMessageParam,
        {
            "role": "tool",
            "tool_call_id": tool_call_id,
            "content": json.dumps(result, ensure_ascii=False),
        },
    )

## 调用模型

In [12]:
def call_llm() -> Any:
    """
    Call LLM once with current messages and tool schemas.

    这里单独封装，是为了让 run_agent 主流程更清晰。
    """
    completion = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice="auto",
    )

    return completion.choices[0].message

## 实现 Mini Agent Loop

In [13]:
def print_tool_execution(execution: ToolExecution) -> None:
    """
    Pretty print one tool execution for notebook debugging.
    """
    print("\n[Tool Call]")
    print("tool_name:", execution["tool_name"])
    print(
        "arguments:",
        json.dumps(
            execution["arguments"],
            ensure_ascii=False,
            indent=2,
        ),
    )
    print(
        "result:",
        json.dumps(
            execution["result"],
            ensure_ascii=False,
            indent=2,
        ),
    )


def run_agent(
    user_input: str,
    *,
    verbose: bool = True,
    max_tool_rounds: int = 3,
) -> str:
    """
    Run a mini agent loop.

    流程：

    1. User input
    2. LLM 判断是否需要 tool
    3. 如果需要，返回 tool_calls
    4. Python 执行 tool
    5. 把 tool result 回传给模型
    6. 模型生成最终回答

    max_tool_rounds 是为了防止模型无限调用工具。
    """
    user_message = cast(
        ChatCompletionMessageParam,
        {
            "role": "user",
            "content": user_input,
        },
    )
    messages.append(user_message)

    for round_index in range(max_tool_rounds + 1):
        assistant_message = call_llm()

        messages.append(assistant_message_to_param(assistant_message))

        tool_calls = getattr(assistant_message, "tool_calls", None)

        # Case 1: No tool call. The model directly gives final answer.
        if not tool_calls:
            final_answer = getattr(assistant_message, "content", None)

            if final_answer is None:
                return ""

            return str(final_answer)

        # Case 2: Model wants to call tools.
        if round_index >= max_tool_rounds:
            return (
                "工具调用轮数超过限制，Agent Loop 已停止。"
                "请检查模型是否陷入了重复调用工具。"
            )

        for tool_call in tool_calls:
            execution = execute_tool_call(tool_call)

            if verbose:
                print_tool_execution(execution)

            tool_message = tool_result_to_message(
                tool_call_id=execution["tool_call_id"],
                result=execution["result"],
            )

            messages.append(tool_message)

    return "Agent Loop ended unexpectedly."

## 基础测试：不需要工具的普通问答

In [14]:
reset_agent_messages()
reset_task_store()

answer = run_agent(
    "什么是 Agent Loop？请用非常简单的话解释。",
    verbose=True,
)

print("\n[Final Answer]")
print(answer)


[Final Answer]
## 什么是 Agent Loop？

**Agent Loop（代理循环）** 简单来说，就是：

**一个 AI 自主思考 → 行动 → 观察结果 → 再思考 → 再行动的循环过程。**

具体流程是这样走的：

1. **思考/推理** 🤔 → AI 分析当前任务，决定下一步做什么
2. **行动/调用工具** 🛠️ → AI 执行一个操作（比如搜索文件、读代码、执行命令）
3. **观察结果** 👀 → AI 看到操作的返回结果
4. **重复循环** 🔁 → 基于新信息，再思考下一步，直到完成任务

---

### 举个生活中的例子 🌰

> 想象你在做一个菜（任务）：
>
> - 你看一眼菜谱 → **思考**
> - 你拿起盐撒一点 → **行动**
> - 你尝一口 → **观察结果**
> - 你觉得淡了，再加点盐 → **再次思考+行动**
> - 再尝 → **观察**
> - 直到味道刚好 → **循环结束**

---

### 在 SWE Agent 中的体现

SWE Agent 就是用这种循环来处理编程任务的：
- 收到一个 Issue（Bug/需求）
- Agent 思考它需要做什么
- 然后调用工具（读文件、编辑代码）
- 看到结果后再决定下一步
- 反复循环，直到修好 Bug 或实现功能

**总结一句话：Agent Loop 就是让 AI 像人一样，不断"想→做→看结果→再想"，步步逼近完成任务。**


## 测试添加任务

In [15]:
answer = run_agent(
    "帮我添加一个任务：完成 3.4 mini agent loop，优先级高。",
    verbose=True,
)

print("\n[Final Answer]")
print(answer)


[Tool Call]
tool_name: add_task
arguments: {
  "title": "完成 3.4 mini agent loop",
  "priority": "high"
}
result: {
  "ok": true,
  "message": "Task added successfully.",
  "task": {
    "id": 1,
    "title": "完成 3.4 mini agent loop",
    "status": "todo",
    "priority": "high"
  }
}

[Final Answer]
任务已经成功添加！来看看详情：

| 字段 | 内容 |
|------|------|
| 📋 **任务 ID** | 1 |
| 📝 **标题** | 完成 3.4 mini agent loop |
| ✅ **状态** | todo（待办） |
| 🔴 **优先级** | 高（high） |

目前该任务还是 **待办（todo）** 状态，需要开始做的时候可以告诉我，我来帮你更新状态为 **doing**，完成后改为 **done**。加油！💪


## 再添加几个任务

In [16]:
print(
    run_agent(
        "帮我添加一个任务：整理 function dispatch 的代码，优先级中等。",
        verbose=True,
    )
)

print(
    run_agent(
        "帮我添加一个任务：复习 tool result 回传模型的流程，优先级高。",
        verbose=True,
    )
)

print(
    run_agent(
        "帮我添加一个任务：写 3.5 的错误处理机制，优先级低。",
        verbose=True,
    )
)


[Tool Call]
tool_name: add_task
arguments: {
  "title": "整理 function dispatch 的代码",
  "priority": "medium"
}
result: {
  "ok": true,
  "message": "Task added successfully.",
  "task": {
    "id": 2,
    "title": "整理 function dispatch 的代码",
    "status": "todo",
    "priority": "medium"
  }
}
任务添加成功！现在你一共有 **2 个任务**：

| ID | 任务标题 | 状态 | 优先级 |
|:--:|----------|:----:|:------:|
| 1 | 完成 3.4 mini agent loop | ⏳ todo | 🔴 高 |
| 2 | 整理 function dispatch 的代码 | ⏳ todo | 🟡 中 |

两个任务目前都是 **待办** 状态，后续需要更新进度的话随时告诉我！😊

[Tool Call]
tool_name: add_task
arguments: {
  "title": "复习 tool result 回传模型的流程",
  "priority": "high"
}
result: {
  "ok": true,
  "message": "Task added successfully.",
  "task": {
    "id": 3,
    "title": "复习 tool result 回传模型的流程",
    "status": "todo",
    "priority": "high"
  }
}
任务添加成功！来看看你现在的 **全部任务清单**：

| ID | 任务标题 | 状态 | 优先级 |
|:--:|----------|:----:|:------:|
| 1 | 完成 3.4 mini agent loop | ⏳ todo | 🔴 高 |
| 2 | 整理 function dispatch 的代码 | ⏳ todo | 🟡 中 |
| 3 | 复习 tool result 回

## 查看当前所有任务

In [17]:
answer = run_agent(
    "我现在有哪些学习任务？",
    verbose=True,
)

print("\n[Final Answer]")
print(answer)


[Tool Call]
tool_name: list_tasks
arguments: {
  "status": "all"
}
result: {
  "ok": true,
  "count": 4,
  "tasks": [
    {
      "id": 1,
      "title": "完成 3.4 mini agent loop",
      "status": "todo",
      "priority": "high"
    },
    {
      "id": 2,
      "title": "整理 function dispatch 的代码",
      "status": "todo",
      "priority": "medium"
    },
    {
      "id": 3,
      "title": "复习 tool result 回传模型的流程",
      "status": "todo",
      "priority": "high"
    },
    {
      "id": 4,
      "title": "写 3.5 的错误处理机制",
      "status": "todo",
      "priority": "low"
    }
  ]
}

[Final Answer]
目前你一共有 **4 个学习任务**，全部处于 **待办（todo）** 状态：

| ID | 任务标题 | 状态 | 优先级 |
|:--:|----------|:----:|:------:|
| 1 | 完成 3.4 mini agent loop | ⏳ todo | 🔴 高 |
| 2 | 整理 function dispatch 的代码 | ⏳ todo | 🟡 中 |
| 3 | 复习 tool result 回传模型的流程 | ⏳ todo | 🔴 高 |
| 4 | 写 3.5 的错误处理机制 | ⏳ todo | 🟢 低 |

**建议推进顺序：** 先做两个高优先级的任务（1 和 3），再做中等优先级的任务 2，最后做低优先级的任务 4。

要开始做哪一个？我可以帮你把状态更新为 **doing**（进行中）😊


## 查看未完成任务

In [18]:
answer = run_agent(
    "我现在还有哪些任务没完成？",
    verbose=True,
)

print("\n[Final Answer]")
print(answer)


[Tool Call]
tool_name: list_tasks
arguments: {
  "status": "all"
}
result: {
  "ok": true,
  "count": 4,
  "tasks": [
    {
      "id": 1,
      "title": "完成 3.4 mini agent loop",
      "status": "todo",
      "priority": "high"
    },
    {
      "id": 2,
      "title": "整理 function dispatch 的代码",
      "status": "todo",
      "priority": "medium"
    },
    {
      "id": 3,
      "title": "复习 tool result 回传模型的流程",
      "status": "todo",
      "priority": "high"
    },
    {
      "id": 4,
      "title": "写 3.5 的错误处理机制",
      "status": "todo",
      "priority": "low"
    }
  ]
}

[Final Answer]
目前 **4 个任务全部未完成**，一个都没做完 😅

| ID | 任务标题 | 状态 | 优先级 |
|:--:|----------|:----:|:------:|
| 1 | 完成 3.4 mini agent loop | ⏳ todo | 🔴 高 |
| 2 | 整理 function dispatch 的代码 | ⏳ todo | 🟡 中 |
| 3 | 复习 tool result 回传模型的流程 | ⏳ todo | 🔴 高 |
| 4 | 写 3.5 的错误处理机制 | ⏳ todo | 🟢 低 |

要不要先挑一个高优先级的开始做？我可以帮你把状态更新为 **doing**（进行中）🚀


## 更新任务状态

In [19]:
answer = run_agent(
    "把 1 号任务标记为完成。",
    verbose=True,
)

print("\n[Final Answer]")
print(answer)


[Tool Call]
tool_name: update_task_status
arguments: {
  "task_id": 1,
  "status": "done"
}
result: {
  "ok": true,
  "message": "Task status updated successfully.",
  "task": {
    "id": 1,
    "title": "完成 3.4 mini agent loop",
    "status": "done",
    "priority": "high"
  }
}

[Final Answer]
✅ **任务 1「完成 3.4 mini agent loop」** 已标记为 **完成（done）**！恭喜！🎉

更新后的任务清单：

| ID | 任务标题 | 状态 | 优先级 |
|:--:|----------|:----:|:------:|
| 1 | ~~完成 3.4 mini agent loop~~ | ✅ **done** | 🔴 高 |
| 2 | 整理 function dispatch 的代码 | ⏳ todo | 🟡 中 |
| 3 | 复习 tool result 回传模型的流程 | ⏳ todo | 🔴 高 |
| 4 | 写 3.5 的错误处理机制 | ⏳ todo | 🟢 低 |

还剩下 **3 个任务**待完成，要不要继续推进下一个？😊


## 搜索任务

In [20]:
answer = run_agent(
    "帮我找一下和 tool 有关的任务。",
    verbose=True,
)

print("\n[Final Answer]")
print(answer)


[Tool Call]
tool_name: search_tasks
arguments: {
  "keyword": "tool"
}
result: {
  "ok": true,
  "keyword": "tool",
  "count": 1,
  "tasks": [
    {
      "id": 3,
      "title": "复习 tool result 回传模型的流程",
      "status": "todo",
      "priority": "high"
    }
  ]
}

[Final Answer]
找到了！与 **"tool"** 相关的任务有 **1 个**：

| ID | 任务标题 | 状态 | 优先级 |
|:--:|----------|:----:|:------:|
| 3 | 复习 tool result 回传模型的流程 | ⏳ todo | 🔴 高 |

这个任务目前还是 **待办** 状态，而且是 **高优先级** 的。要不要现在就开始做它？我可以帮你更新状态为 **doing**（进行中）😊


## 直接查看本地状态

In [21]:
get_all_tasks_as_dicts()

[{'id': 1,
  'title': '完成 3.4 mini agent loop',
  'status': 'done',
  'priority': 'high'},
 {'id': 2,
  'title': '整理 function dispatch 的代码',
  'status': 'todo',
  'priority': 'medium'},
 {'id': 3,
  'title': '复习 tool result 回传模型的流程',
  'status': 'todo',
  'priority': 'high'},
 {'id': 4, 'title': '写 3.5 的错误处理机制', 'status': 'todo', 'priority': 'low'}]

## 查看完整 messages

In [22]:
for index, message in enumerate(messages):
    print(f"\n--- Message {index} ---")
    print(json.dumps(message, ensure_ascii=False, indent=2))


--- Message 0 ---
{
  "role": "system",
  "content": "You are a Mini Study Task Agent.\n\nYour job is to help the user manage study tasks related to SWE Agent learning.\n\nYou have access to task management tools:\n- add_task\n- list_tasks\n- update_task_status\n- search_tasks\n\nRules:\n1. If the user asks conceptual questions, answer directly without using tools.\n2. If the user wants to add, list, update, or search tasks, use the appropriate tool.\n3. Do not pretend that a task was added, updated, or found unless the tool result confirms it.\n4. After receiving tool results, explain the result clearly in Chinese.\n5. Keep the final answer concise and practical."
}

--- Message 1 ---
{
  "role": "user",
  "content": "什么是 Agent Loop？请用非常简单的话解释。"
}

--- Message 2 ---
{
  "role": "assistant",
  "content": "## 什么是 Agent Loop？\n\n**Agent Loop（代理循环）** 简单来说，就是：\n\n**一个 AI 自主思考 → 行动 → 观察结果 → 再思考 → 再行动的循环过程。**\n\n具体流程是这样走的：\n\n1. **思考/推理** 🤔 → AI 分析当前任务，决定下一步做什么\n2. **行动/调用工具** 🛠️ → AI 执行一个操

## END